# SETUP & CHẠY PIPELINE DỰ ĐOÁN CTR TRÊN AZURE DATABRICKS
---
## PHẦN 1: HƯỚNG DẪN SETUP HẠ TẦNG AZURE & VS CODE

### 1. Trên Azure Portal (Giao diện Web)
- **Bước 1:** Đảm bảo đã có 1 **Storage Account** (Bật tính năng *Hierarchical namespace* để thành Data Lake Gen2). Tạo 1 Container tên `raw-data` và up data lên đó.
- **Bước 2:** Vào Azure Databricks, tạo 1 Workspace và bật 1 Cluster (Cụm máy chủ)
- **Bước 3:** Lấy Access Key của Storage Account (để load data) và tạo Personal Access Token (PAT) trong phần Settings của Databricks (để VS Code kết nối vào).

### 2. Trên VS Code (Máy tính cá nhân)
- **Bước 1:** Cài đặt Extension **Databricks** (chính chủ Microsoft).
- **Bước 2:** Bấm vào logo Databricks ở thanh menu trái -> Chọn *Configure Workspace*.
- **Bước 3:** Nhập URL của Databricks Workspace và dán mã Token (PAT) vào để xác thực.
- **Bước 4:** Gõ `Databricks: Select Cluster` và chọn đúng cụm Cluster đang bật trên web.
- **Bước 5:** Bật Sync (Biểu tượng 2 mũi tên xoay vòng) để code tự động đồng bộ lên cloud.
- **Bước 6:** Khi chạy code, KHÔNG bấm nút Play thường, mà bấm nút **Run File as Workflow on Databricks** 

---
## PHẦN 2: CHẠY PIPELINE DỮ LIỆU & HUẤN LUYỆN MÔ HÌNH
### Bước A: Cấp quyền và Đọc dữ liệu từ Cloud (Data Ingestion)

In [ ]:
from pyspark.sql import SparkSession

# Khởi tạo Spark Session 
spark = SparkSession.builder.appName("CTR_Prediction_Pipeline").getOrCreate()

# 1. Điền thông tin kho lưu trữ 
storage_account_name = "ctrdatalake2026" 
storage_account_access_key = "ĐIỀN_ACCESS_KEY_VÀO_ĐÂY" 
container_name = "raw-data"
file_name = "test.txt" 

# 2. Đưa chìa khóa cho Spark để kết nối an toàn
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_access_key
)

# 3. Tạo đường dẫn URI
file_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{file_name}"

# 4. Đọc dữ liệu 
print("[INFO] Đang kéo dữ liệu từ Azure Data Lake...")
df = spark.read.csv(file_path, sep='\t', header=True, inferSchema=True)

print("[INFO] Hoàn thành load dữ liệu! Xem 5 dòng đầu:")
display(df)

### Bước B: Tiền xử lý dữ liệu và Feature Engineering
Chúng ta sẽ loại bỏ dữ liệu rỗng và chuyển đổi các cột dạng chữ (Categorical) thành dạng số (Vector) bằng Pipeline để đưa vào mô hình học máy.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler

print("[INFO] Bắt đầu dọn dẹp và biến đổi dữ liệu...")

# 1. Xóa các dòng rỗng (NaN/Null)
df_clean = df.dropna()

try:
    stages = []
    # 2. Chuyển chữ thành số cho các cột Categorical
    for cat_col in categorical_columns:
        indexer = StringIndexer(inputCol=cat_col, outputCol=cat_col + "_index", handleInvalid="keep")
        stages.append(indexer)
    
    # 3. Gom tất cả các cột thành 1 cột duy nhất tên là 'features'
    assembler_inputs = [c + "_index" for c in categorical_columns] + numerical_columns
    assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="skip")
    stages.append(assembler)
    
    # 4. Chạy Pipeline
    pipeline = Pipeline(stages=stages)
    pipeline_model = pipeline.fit(df_clean)
    df_transformed = pipeline_model.transform(df_clean)
    
    # 5. Chia tập Train 80% / Test 20%
    train_data, test_data = df_transformed.randomSplit([0.8, 0.2], seed=42)
    print("[INFO] Tiền xử lý thành công! Đã sẵn sàng Train mô hình.")

except Exception as e:
    print("\n[CẢNH BÁO] Chưa thể xử lý vì tên cột ở trên chưa khớp với file data thực tế.")
    print(f"Lỗi chi tiết: {e}")

### Bước C: Huấn luyện mô hình phân loại (Logistic Regression)
Logistic Regression được sử dụng để tối ưu hóa việc phân loại nhị phân (Click hay Không Click) vì nó hoạt động rất nhanh trên dữ liệu phân tán (Spark).

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

try:
    print("[INFO] Bắt đầu huấn luyện mô hình (Training)...")
    # 1. Khai báo mô hình
    lr = LogisticRegression(featuresCol='features', labelCol=label_col, maxIter=10)
    
    # 2. Huấn luyện với tập Train
    lr_model = lr.fit(train_data)
    
    # 3. Dự đoán trên tập Test
    predictions = lr_model.transform(test_data)
    
    # 4. Đánh giá độ chính xác (Area Under ROC)
    evaluator = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    auc_score = evaluator.evaluate(predictions)
    
    print(f"\n🚀 [HOÀN THÀNH] Độ chính xác của mô hình (AUC ROC): {auc_score:.4f}")
    print("Vài dòng kết quả dự đoán:")
    predictions.select(label_col, "prediction", "probability").show(5)

except Exception as e:
    print("[CẢNH BÁO] Vui lòng cấu hình tên cột ở Bước B trước khi chạy Bước C.")

### Bước C: Huấn luyện mô hình phân loại (Logistic Regression)
Logistic Regression được sử dụng để tối ưu hóa việc phân loại nhị phân (Click hay Không Click) vì nó hoạt động rất nhanh trên dữ liệu phân tán (Spark).

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

try:
    print("[INFO] Bắt đầu huấn luyện mô hình (Training)...")
    # 1. Khai báo mô hình
    lr = LogisticRegression(featuresCol='features', labelCol=label_col, maxIter=10)
    
    # 2. Huấn luyện với tập Train
    lr_model = lr.fit(train_data)
    
    # 3. Dự đoán trên tập Test
    predictions = lr_model.transform(test_data)
    
    # 4. Đánh giá độ chính xác (Area Under ROC)
    evaluator = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    auc_score = evaluator.evaluate(predictions)
    
    print(f"\n🚀 [HOÀN THÀNH] Độ chính xác của mô hình (AUC ROC): {auc_score:.4f}")
    print("Vài dòng kết quả dự đoán:")
    predictions.select(label_col, "prediction", "probability").show(5)

except Exception as e:
    print("[CẢNH BÁO] Vui lòng cấu hình tên cột ở Bước B trước khi chạy Bước C.")